# Build a Complete Vision Pipeline — Capstone Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Data contracts

In [ ]:
```python

from pydantic import BaseModel, Field

from typing import List, Optional, Tuple

class Detection(BaseModel):

    box: Tuple[float, float, float, float]

    score: float = Field(ge=0, le=1)

    class_id: int = Field(ge=0)

    mask_rle: Optional[str] = None

class Classification(BaseModel):

    detection_index: int

    class_id: int

    class_name: str

    score: float = Field(ge=0, le=1)

class PipelineResult(BaseModel):

    image_id: str

    detections: List[Detection]

    classifications: List[Classification]

    inference_ms: float

In [ ]:
```

Five seconds of code saves an hour of debugging on any serious pipeline.

### Step 2: A minimal Pipeline class

In [ ]:
```python

import time

import numpy as np

import torch

from PIL import Image

class VisionPipeline:

    def __init__(self, detector, classifier, class_names,

                 device="cpu", min_crop=32):

        self.detector = detector.to(device).eval()

        self.classifier = classifier.to(device).eval()

        self.class_names = class_names

        self.device = device

        self.min_crop = min_crop

    def preprocess(self, image):

        """

        image: PIL.Image or np.ndarray (H, W, 3) uint8

        returns: CHW float tensor on device

        """

        if isinstance(image, Image.Image):

            image = np.asarray(image.convert("RGB"))

        tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        return tensor.to(self.device)

    @torch.no_grad()

    def detect(self, image_tensor):

        return self.detector([image_tensor])[0]

    @torch.no_grad()

    def classify(self, crops):

        if len(crops) == 0:

            return []

        batch = torch.stack(crops).to(self.device)

        logits = self.classifier(batch)

        probs = logits.softmax(-1)

        scores, cls = probs.max(-1)

        return list(zip(cls.tolist(), scores.tolist()))

    def run(self, image, image_id="anonymous"):

        t0 = time.perf_counter()

        tensor = self.preprocess(image)

        det = self.detect(tensor)

        crops = []

        detections = []

        valid_indices = []

        for i, (box, score, cls) in enumerate(zip(det["boxes"], det["scores"], det["labels"])):

            x1, y1, x2, y2 = [max(0, int(b)) for b in box.tolist()]

            x2 = min(x2, tensor.shape[-1])

            y2 = min(y2, tensor.shape[-2])

            detections.append(Detection(

                box=(x1, y1, x2, y2),

                score=float(score),

                class_id=int(cls),

            ))

            if (x2 - x1) < self.min_crop or (y2 - y1) < self.min_crop:

                continue

            crop = tensor[:, y1:y2, x1:x2]

            crop = torch.nn.functional.interpolate(

                crop.unsqueeze(0),

                size=(224, 224),

                mode="bilinear",

                align_corners=False,

            )[0]

            crops.append(crop)

            valid_indices.append(i)

        class_preds = self.classify(crops)

        classifications = []

        for valid_idx, (cls_id, cls_score) in zip(valid_indices, class_preds):

            classifications.append(Classification(

                detection_index=valid_idx,

                class_id=int(cls_id),

                class_name=self.class_names[cls_id],

                score=float(cls_score),

            ))

        return PipelineResult(

            image_id=image_id,

            detections=detections,

            classifications=classifications,

            inference_ms=(time.perf_counter() - t0) * 1000,

        )

In [ ]:
```

Every interface is typed. Every failure path has a specific handling decision.

### Step 3: Wire a detector and a classifier

In [ ]:
```python

from torchvision.models.detection import maskrcnn_resnet50_fpn_v2

from torchvision.models import convnext_tiny

# Use ImageNet-pretrained weights for a realistic pipeline without training

detector = maskrcnn_resnet50_fpn_v2(weights="DEFAULT")

classifier = convnext_tiny(weights="DEFAULT")

class_names = [f"imagenet_class_{i}" for i in range(1000)]

pipe = VisionPipeline(detector, classifier, class_names)

# Smoke test with a synthetic image

test_image = (np.random.rand(400, 600, 3) * 255).astype(np.uint8)

result = pipe.run(test_image, image_id="demo")

print(result.model_dump_json(indent=2)[:500])

In [ ]:
```

### Step 4: FastAPI service

In [ ]:
```python

from fastapi import FastAPI, UploadFile, HTTPException

from io import BytesIO

app = FastAPI()

pipe = None  # initialised on startup

@app.on_event("startup")

def load():

    global pipe

    detector = maskrcnn_resnet50_fpn_v2(weights="DEFAULT").eval()

    classifier = convnext_tiny(weights="DEFAULT").eval()

    pipe = VisionPipeline(detector, classifier, class_names=[f"c{i}" for i in range(1000)])

@app.post("/detect")

async def detect_endpoint(file: UploadFile):

    if file.content_type not in {"image/jpeg", "image/png", "image/webp"}:

        raise HTTPException(status_code=400, detail="unsupported image type")

    data = await file.read()

    try:

        img = Image.open(BytesIO(data)).convert("RGB")

    except Exception:

        raise HTTPException(status_code=400, detail="cannot decode image")

    result = pipe.run(img, image_id=file.filename or "upload")

    return result.model_dump()

In [ ]:
```

Run with `uvicorn main:app --host 0.0.0.0 --port 8000`. Test with `curl -F 'file=@dog.jpg' http://localhost:8000/detect`.

### Step 5: Benchmark the pipeline

In [ ]:
```python

import time

def benchmark(pipe, num_runs=20, image_size=(400, 600)):

    img = (np.random.rand(*image_size, 3) * 255).astype(np.uint8)

    pipe.run(img)  # warm up

    stages = {"preprocess": [], "detect": [], "classify": [], "total": []}

    for _ in range(num_runs):

        t0 = time.perf_counter()

        tensor = pipe.preprocess(img)

        t1 = time.perf_counter()

        det = pipe.detect(tensor)

        t2 = time.perf_counter()

        crops = []

        for box in det["boxes"]:

            x1, y1, x2, y2 = [max(0, int(b)) for b in box.tolist()]

            x2 = min(x2, tensor.shape[-1])

            y2 = min(y2, tensor.shape[-2])

            if (x2 - x1) >= pipe.min_crop and (y2 - y1) >= pipe.min_crop:

                crop = tensor[:, y1:y2, x1:x2]

                crop = torch.nn.functional.interpolate(

                    crop.unsqueeze(0), size=(224, 224), mode="bilinear", align_corners=False

                )[0]

                crops.append(crop)

        pipe.classify(crops)

        t3 = time.perf_counter()

        stages["preprocess"].append((t1 - t0) * 1000)

        stages["detect"].append((t2 - t1) * 1000)

        stages["classify"].append((t3 - t2) * 1000)

        stages["total"].append((t3 - t0) * 1000)

    for stage, times in stages.items():

        times.sort()

        print(f"{stage:12s}  p50={times[len(times)//2]:7.1f} ms  p95={times[int(len(times)*0.95)]:7.1f} ms")

In [ ]:
```

Typical output on CPU: preprocess ~3 ms, detect 300-500 ms, classify 20-40 ms, total 350-550 ms. On GPU, detect is 20-40 ms and the preprocess + classify start to matter more in relative terms.

## Exercises

In [ ]:
1. **(Easy)** Run the pipeline on 10 images from any open dataset. Report the average time per stage and the distribution of detection counts per image.
2. **(Medium)** Add a mask output field to `Detection` and encode it as RLE. Verify the JSON stays under 1MB even for a 10-object image.
3. **(Hard)** Add a micro-batcher in front of the classifier: collect crops for up to 10 ms, classify them all in one GPU call, return results per request. Measure the throughput gain at 5 concurrent requests per second and the latency added.